# Introduction to Panel Data Methods in Python

*A beginner-friendly tour of seven panel-data estimators — POLS, Between, First-Differences, Fixed Effects, Two-Way FE, Random Effects, and Correlated Random Effects (Mundlak) — applied to a two-period worker wage panel.*

## 1. Overview

Imagine you have data on the same workers in two different years — 2010 and 2012 — and you want to know whether *joining a union* raises a worker's wage. A simple regression on the pooled data says yes, by about 7.5%. But that headline number hides a problem that has occupied econometricians for fifty years: workers who join unions are not the same as workers who don't. Maybe they have less formal education, or they work in industries where unions are common, or they are older and have negotiated harder. If any of those *unobserved* differences also affect wages, the 7.5% estimate is mixing the union effect with everything else that comes bundled with union status.

This is the **omitted-variable bias** problem, and panel data — repeated observations on the same units over time — gives us several ways to fight it. By comparing each worker to *themselves* across years, we can strip out anything that is constant within a person (innate ability, gender, schooling, family background) and isolate the effect of switching union status. The price is a much smaller effective sample: only the workers who actually changed union status between 2010 and 2012 contribute to the estimate. The benefit is a coefficient that is much harder to dismiss as confounded.

This tutorial walks through the seven canonical panel estimators on a real two-period wage panel: pooled OLS, between, first-differences, the within (fixed effects) estimator, two-way fixed effects, random effects, and Mundlak's correlated random effects. Along the way we run the Hausman test and visualize what the *within transformation* actually does to the data. The headline result will surprise some readers: once we account for unobserved worker traits, the union wage premium roughly *triples* — from about 7% to about 21%.

**Learning objectives:**

- Understand the difference between *between* and *within* variation in panel data, and why this distinction drives the choice of estimator.
- Implement seven panel-data estimators in Python using `pyfixest` and `linearmodels`, with one short code block per method.
- Visualize the within transformation and see geometrically why fixed effects produce a different slope than pooled OLS.
- Run the Hausman test to compare fixed and random effects, and use the Mundlak/CRE specification as the modern alternative.
- Interpret the factor-of-three gap between cross-sectional and within estimators in terms of selection on unobservables.

> **Note for Colab readers.** The blog post version of this tutorial includes a Mermaid flowchart showing the estimator family and how Hausman and Mundlak choose between FE and RE. Mermaid does not render natively in Colab, so the diagram is omitted here; see the [post](https://carlos-mendez.org/post/python_panel_intro/) for the visual.

## 2. Setup and imports

We use [`pyfixest`](https://pyfixest.org/) for OLS and absorbed fixed effects, [`linearmodels`](https://bashtage.github.io/linearmodels/panel/introduction.html) for the random-effects GLS estimator, and `scipy.stats.chi2` for the Hausman test critical value. Install the two packages that are not already in Colab:

In [ ]:
!pip install pyfixest linearmodels -q

Standard imports plus the random-effects estimator and the chi-square distribution. Site palette colors are defined for the dark-navy figure theme used throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyfixest as pf
import statsmodels.api as sm
from linearmodels.panel import RandomEffects
from scipy.stats import chi2

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

# Site palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"
LIGHT_BLUE = "#8FB4D8"

# Dark theme
DARK_NAVY = "#0f1729"
GRID_LINE = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

METHOD_COLORS = {
    "POLS":    "#999999",
    "Between": LIGHT_BLUE,
    "FDFE":    STEEL_BLUE,
    "FE":      WARM_ORANGE,
    "TWFE":    WARM_ORANGE,
    "RE":      TEAL,
    "CRE":     "#c4623d",
}

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

## 3. Data loading

We load a two-period wage panel from a Stata `.dta` file: NLSY-style data on US workers observed in 2010, 2012, 2014, 2016, and 2018. For pedagogical clarity we restrict the analysis to **2010 and 2012 only**, which makes T = 2 and gives us the cleanest possible illustration of the textbook result that first-differences and the within estimator are the same thing.

In [ ]:
DATA_URL = "https://github.com/quarcs-lab/data-open/raw/master/isds/wage_panel_bob4.dta"
df_full = pd.read_stata(DATA_URL)

# Keep two periods so the FD = Within identity is visible.
df = df_full[df_full["year"].isin([2010, 2012])].copy()
df = df.sort_values(["ID", "year"]).reset_index(drop=True)

# Convert union "Yes/No" to 1/0; build a female dummy.
df["union"] = df["union"].map({"Yes": 1, "No": 0, 1: 1, 0: 0})
df["female"] = (df["gender"].astype(str).str.strip().str.lower() == "female").astype(float)

# Coerce numeric columns (pd.read_stata leaves some as Categorical).
for col in ["lwage", "age", "schooling", "ID", "year", "union"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with missing values in the variables we use.
df = df.dropna(subset=["lwage", "union", "age", "schooling"]).reset_index(drop=True)
df["year"] = df["year"].astype(int)

print(f"Filtered panel: {df.shape[0]} rows × {df.shape[1]} cols")

The next block prints panel structure and descriptive statistics. The "balanced" check confirms every worker has exactly two observations.

In [ ]:
print(f"Individuals (N): {df['ID'].nunique()}")
print(f"Time periods (T): {df['year'].nunique()}")
print(f"Observations (N×T): {len(df)}")
print(f"Balanced: {(df.groupby('ID')['year'].count() == df['year'].nunique()).all()}")
print()
print(df[["lwage", "union", "age", "schooling"]].describe().round(4))

**Interpretation.** The analysis sample is a perfectly balanced panel of 2,199 prime-age workers (mean age 35.7, range 25–49) observed in 2010 and 2012, for 4,398 worker-year observations. Only 16.3% of the sample is unionized in any given period (mean union = 0.1626), which means the dataset leans heavily on non-union workers — a relevant constraint for any estimator that uses cross-sectional variation. With balanced T = 2, the within and first-difference transformations are particularly clean because every individual contributes the same amount of within-variation: exactly one switch (or non-switch) per regressor.

## 4. Between vs within variance: how much do panel methods have to work with?

Before estimating anything, it helps to ask a diagnostic question: for each variable, how much variation comes from differences *between* workers and how much from changes *within* workers over time? Fixed-effects estimators only use the within part. If the within part is tiny, FE will be noisy no matter how large the sample is.

The decomposition splits each variable's variance into two pieces. The **between** part is the variance of each worker's two-year mean: $\mathrm{Var}(\bar{x}_i)$. The **within** part is the variance of each observation around its own worker's mean: $\mathrm{Var}(x_{it} - \bar{x}_i)$. Their sum is (approximately) the total variance.

In [ ]:
variation_data = []
for var in ["lwage", "union", "age", "schooling"]:
    overall_sd = df[var].std()
    between_sd = df.groupby("ID")[var].mean().std()
    within_sd = (df[var] - df.groupby("ID")[var].transform("mean")).std()
    between_pct = between_sd**2 / (between_sd**2 + within_sd**2) * 100
    print(f"{var:<10} overall {overall_sd:.4f}  between {between_sd:.4f}"
          f"  within {within_sd:.4f}  between% {between_pct:.1f}")
    variation_data.append({
        "variable": var,
        "between_pct": round(between_pct, 1),
        "within_pct":  round(100 - between_pct, 1),
    })

variation_df = pd.DataFrame(variation_data)

In [ ]:
# Figure 1 — between vs within shares.
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_linewidth(0)
y_pos = np.arange(len(variation_df))
between_vals = variation_df["between_pct"].values
within_vals = variation_df["within_pct"].values

ax.barh(y_pos, between_vals, 0.5, label="Between (cross-individual)",
        color=STEEL_BLUE, edgecolor=DARK_NAVY, linewidth=0.5)
ax.barh(y_pos, within_vals, 0.5, left=between_vals,
        label="Within (over time)",
        color=WARM_ORANGE, edgecolor=DARK_NAVY, linewidth=0.5)

for i, (b, w) in enumerate(zip(between_vals, within_vals)):
    if b > 8:
        ax.text(b/2, i, f"{b:.0f}%", ha="center", va="center",
                fontsize=11, fontweight="bold", color=WHITE_TEXT)
    if w > 8:
        ax.text(b + w/2, i, f"{w:.0f}%", ha="center", va="center",
                fontsize=11, fontweight="bold", color=WHITE_TEXT)

ax.set_yticks(y_pos)
ax.set_yticklabels([v.title() for v in variation_df["variable"]],
                   fontsize=12, color=LIGHT_TEXT)
ax.set_xlabel("Share of Total Variance (%)", fontsize=13, color=LIGHT_TEXT)
ax.set_title("Between vs Within Variation in Panel Data",
             fontsize=15, fontweight="bold", color=WHITE_TEXT)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=11)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

**Interpretation.** Almost all of the variation in our variables is *between* workers, not over time within a worker. Union status is 93.9% between and only 9.1% within — fixed-effects estimators have access to that thin 9% slice of total union variance. Schooling has zero within-variation (100% between) because nobody's reported education changes between 2010 and 2012 in this sample, which is why FE will mechanically drop schooling from the regression. The big methodological consequence is that FE standard errors will be much larger than POLS standard errors, so the choice between FE and RE is not just a question of unbiasedness; it is also a question of statistical precision.

## 5. Visualizing the panel: who actually changes union status?

The variance decomposition tells us the within share is small. A spaghetti plot of individual log-wage trajectories makes the same point visually. We sample 30 random workers and color each line by the worker's union pattern: orange if always union, blue if never union, and teal if union status changed between 2010 and 2012.

In [ ]:
sample_ids = rng.choice(df["ID"].unique(), size=30, replace=False)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)

for pid in sample_ids:
    person = df[df["ID"] == pid].sort_values("year")
    if person["union"].nunique() > 1:
        ax.plot(person["year"], person["lwage"], "o-", color=TEAL,
                alpha=0.85, linewidth=2, markersize=6, zorder=3)
    else:
        c = WARM_ORANGE if person["union"].iloc[0] == 1 else STEEL_BLUE
        ax.plot(person["year"], person["lwage"], "o-", color=c,
                alpha=0.35, linewidth=1, markersize=4)

legend_elements = [
    Line2D([0], [0], color=STEEL_BLUE, alpha=0.6, lw=1.5, marker="o",
           markersize=5, label="Never union"),
    Line2D([0], [0], color=WARM_ORANGE, alpha=0.6, lw=1.5, marker="o",
           markersize=5, label="Always union"),
    Line2D([0], [0], color=TEAL, lw=2.5, marker="o",
           markersize=6, label="Union status changed"),
]
ax.legend(handles=legend_elements, loc="upper center",
          bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=11)
ax.set_xlabel("Year", fontsize=13, color=LIGHT_TEXT)
ax.set_ylabel("Log Wage", fontsize=13, color=LIGHT_TEXT)
ax.set_title("Individual Wage Trajectories (30 sampled workers)",
             fontsize=15, fontweight="bold", color=WHITE_TEXT)
ax.set_xticks(sorted(df["year"].unique()))
plt.tight_layout()
plt.show()

**Interpretation.** Most of the lines are flat-colored (blue or orange): workers who are *always* or *never* in a union over the two-year window. Only the teal lines — the ones that change union status — provide identifying information for fixed effects, first-differences, and Mundlak/CRE. If you squint at the figure and ignore the teal lines, you have effectively run a between estimator. If you ignore everything except the teal lines, you have run fixed effects.

## 6. Pooled OLS: the naive baseline

We start with the simplest possible estimator: regress log wages on union membership, treating every worker-year as if it were an independent observation. This is **pooled OLS** (POLS). It ignores the panel structure entirely.

In [ ]:
# Stata: reg lwage union, robust
fit_pols = pf.feols("lwage ~ union", data=df, vcov="HC1")
pols_coef = fit_pols.coef()["union"]
pols_se = fit_pols.se()["union"]
print(f"Union coefficient: {pols_coef:.4f}  (SE {pols_se:.4f})")

**Interpretation.** Pooled OLS reports a union wage premium of 7.5 log points (SE 2.3 percentage points), highly significant at conventional levels (t ≈ 3.25). This is the textbook cross-sectional answer and the number a naive analyst would report. It is almost certainly biased: if higher-ability workers select *out of* unionized jobs, then POLS confounds the union effect with whatever ability does to wages.

## 7. Between estimator: the cross-sectional benchmark

The **between estimator** takes POLS to its logical extreme: collapse each worker to their two-year mean, then run OLS across workers. This uses *only* between-individual variation — the mirror image of fixed effects.

In [ ]:
# Stata: xtreg lwage union, be
df_between = df.groupby("ID")[["lwage", "union"]].mean().reset_index()
fit_between = pf.feols("lwage ~ union", data=df_between, vcov="HC1")
between_coef = fit_between.coef()["union"]
between_se = fit_between.se()["union"]
print(f"Union coefficient: {between_coef:.4f}  (SE {between_se:.4f})")
print(f"Sample collapsed to {len(df_between)} individual averages.")

**Interpretation.** Collapsing the panel to 2,199 individual averages and running OLS gives 6.6 log points (SE 3.1) — close to POLS (0.075) because 94% of union variance is between-worker. Both Between and POLS share the same identification problem and serve as the *pre-FE* benchmarks against which the within-style estimators will diverge sharply.

## 8. First-differences: subtracting the past from the present

The first within-style estimator we will see is **first-differences** (FDFE). The idea is to subtract each worker's 2010 values from their 2012 values; any time-invariant trait cancels out in the subtraction. We are left with a regression of $\Delta\mathrm{lwage}$ on $\Delta\mathrm{union}$.

Formally, write the panel model as

$$y_{it} = \alpha_i + \beta x_{it} + u_{it}$$

where $\alpha_i$ is the worker-specific (unobserved) effect. Differencing across the two periods gives

$$y_{i,2012} - y_{i,2010} = \beta\,(x_{i,2012} - x_{i,2010}) + (u_{i,2012} - u_{i,2010})$$

In words, this says: the change in wages between 2010 and 2012 equals $\beta$ times the change in union status, plus a noise term. The worker-specific $\alpha_i$ has vanished. Mapping to code: $y$ is the `lwage` column, $x$ is `union`, $\alpha_i$ is whatever is unique about each worker's `ID`, and $\beta$ is the parameter we want to estimate.

In [ ]:
# Stata: bysort ID: gen d_lwage = lwage - L.lwage; reg d_lwage d_union, robust
df_diff = (df.sort_values(["ID", "year"])
             .groupby("ID")[["lwage", "union"]].diff().dropna())
df_diff.columns = ["d_lwage", "d_union"]

fit_fdfe = pf.feols("d_lwage ~ d_union", data=df_diff, vcov="HC1")
fdfe_coef = fit_fdfe.coef()["d_union"]
fdfe_se = fit_fdfe.se()["d_union"]
print(f"Union coefficient: {fdfe_coef:.4f}  (SE {fdfe_se:.4f})")
print(f"Differenced sample: {len(df_diff)} rows (one per worker since T=2).")

**Interpretation.** The first-difference estimator returns 21.1 log points (SE 7.9), almost three times the POLS estimate, with a 95% confidence interval of roughly [0.06, 0.37]. The standard error is about 3.4× larger — the classic signature of moving from a cross-sectional design to a switcher-only design. The CI is wide but excludes zero, so the upward revision is statistically detectable.

## 9. Within / Fixed effects: the same idea, run differently

The **within estimator** (also called fixed effects, FE) achieves the same goal as first-differences through a different transformation: it subtracts each worker's *mean* from each observation. Every variable becomes $\tilde{x}_{it} = x_{it} - \bar{x}_i$. After this *within transformation*, OLS on the demeaned data delivers the FE coefficient. Modern software (`pyfixest` here, `reghdfe` in Stata) hides the demeaning step and just lets us write `lwage ~ union | ID`.

In [ ]:
# Manual demeaning — pedagogical, makes the within transformation visible.
df["lwage_demean"] = df["lwage"] - df.groupby("ID")["lwage"].transform("mean")
df["union_demean"] = df["union"] - df.groupby("ID")["union"].transform("mean")

# Stata: xtreg lwage union, fe robust
fit_fe = pf.feols("lwage ~ union | ID", data=df, vcov="HC1")
fe_coef = fit_fe.coef()["union"]
fe_se = fit_fe.se()["union"]
print(f"Union coefficient: {fe_coef:.4f}  (SE {fe_se:.4f})")

# DVFE: explicit individual dummies — same coefficient, but estimates N-1 nuisance intercepts.
df["ID_str"] = df["ID"].astype(str)
fit_dvfe = pf.feols("lwage ~ union + C(ID_str)", data=df, vcov="HC1")
print(f"DVFE coefficient: {fit_dvfe.coef()['union']:.4f}  (same as FE)")

The figure below visualizes what the demeaning actually does. The left panel shows the raw data — union (jittered) on the x-axis, log wage on the y-axis, and a POLS regression line. The right panel shows the same observations after subtracting each worker's mean from both variables; the FE regression line goes through the demeaned cloud and the origin.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
fig.patch.set_linewidth(0)

jitter_raw = rng.normal(0, 0.025, size=len(df))
jitter_dem = rng.normal(0, 0.005, size=len(df))

# Left — raw data, POLS slope.
ax1.scatter(df["union"] + jitter_raw, df["lwage"],
            s=10, alpha=0.30, color=STEEL_BLUE, edgecolors="none")
b_pols, a_pols = np.polyfit(df["union"], df["lwage"], 1)
xs = np.array([-0.1, 1.1])
ax1.plot(xs, a_pols + b_pols * xs, color=WHITE_TEXT, lw=2)
ax1.set_xlabel("Union (raw)", fontsize=12, color=LIGHT_TEXT)
ax1.set_ylabel("Log wage (raw)", fontsize=12, color=LIGHT_TEXT)
ax1.set_title(f"Raw data — POLS slope = {b_pols:.3f}",
              fontsize=13, color=WHITE_TEXT)

# Right — demeaned data, FE slope.
ax2.scatter(df["union_demean"] + jitter_dem, df["lwage_demean"],
            s=10, alpha=0.30, color=WARM_ORANGE, edgecolors="none")
b_fe = np.polyfit(df["union_demean"], df["lwage_demean"], 1)[0]
xr = np.array([df["union_demean"].min(), df["union_demean"].max()])
ax2.plot(xr, b_fe * xr, color=WHITE_TEXT, lw=2)
ax2.axhline(0, color=LIGHT_TEXT, lw=0.5, ls="--", alpha=0.5)
ax2.axvline(0, color=LIGHT_TEXT, lw=0.5, ls="--", alpha=0.5)
ax2.set_xlabel("Union − mean(Union)", fontsize=12, color=LIGHT_TEXT)
ax2.set_ylabel("Log wage − mean(Log wage)", fontsize=12, color=LIGHT_TEXT)
ax2.set_title(f"Demeaned data — FE slope = {b_fe:.3f}",
              fontsize=13, color=WHITE_TEXT)

fig.suptitle("Within Transformation: Removing Individual Means",
             fontsize=15, fontweight="bold", color=WHITE_TEXT, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation.** The two panels look almost like different datasets, but they come from the *same* observations. On the left (raw data), the POLS slope is ≈ 0.08, dragged down by the union and non-union workers' mean wages being close to each other. On the right (demeaned data), the FE slope is ≈ 0.21, identified only by the workers who actually changed union status — those are the points that move off the origin. The within slope is steeper because we are no longer comparing *across* workers (where ability and schooling confound the picture); we are comparing each worker to themselves.

The FE coefficient of 0.2103 is essentially identical to FDFE (0.2113). The tiny gap of +0.001 comes from the fact that our FD regression includes an intercept (which absorbs an aggregate time trend), while plain FE does not. Once we add a year fixed effect to FE — that's two-way FE in the next section — the gap closes exactly.

## 10. Two-way fixed effects: closing the FD–FE gap

**Two-way fixed effects** (TWFE) absorbs both individual and time effects. We let `pyfixest` handle both with `| ID + year`. This is the workhorse specification of applied micro and DID research.

In [ ]:
# Stata: reghdfe lwage union age, absorb(ID year) vce(cluster ID)
fit_twfe = pf.feols("lwage ~ union + age | ID + year", data=df, vcov={"CRV1": "ID"})
twfe_coef = fit_twfe.coef()["union"]
twfe_se = fit_twfe.se()["union"]
print(f"Union coefficient: {twfe_coef:.4f}  (SE {twfe_se:.4f})")

**Interpretation.** TWFE returns 21.3 log points (SE 7.9), almost indistinguishable from FE (0.210). The small +0.002 gap relative to FE is exactly what closes the FD–FE puzzle: by absorbing year effects we remove the aggregate wage trend that FD's intercept was capturing. Schooling, gender, and any other time-invariant regressor would be silently absorbed by the individual fixed effects — you cannot identify the effect of something that does not change within a worker.

## 11. Random effects: betting on the no-correlation assumption

The **random-effects** (RE) estimator takes a different stance: it treats the worker effect $\alpha_i$ as a *random* draw from a population, *uncorrelated with the regressors*. If that assumption holds, RE is more efficient than FE because it uses both within and between variation. If the assumption fails, RE is biased.

Two pieces of vocabulary that the rest of this section relies on. First, RE is fit by *generalized least squares* (GLS) — a weighted regression that downweights observations whose individual effect is harder to learn from, which is what lets RE blend between- and within-variation in the right proportions. Second, an estimator is *consistent* if its bias shrinks toward zero as the sample grows; an *inconsistent* estimator stays biased no matter how much data you collect. RE is consistent under the no-correlation assumption; FE is consistent under weaker assumptions and is therefore the safer default whenever the no-correlation assumption is suspect.

In [ ]:
# Stata: xtreg lwage union, re robust
df_re = df.set_index(["ID", "year"])
exog = sm.add_constant(df_re[["union"]])
fit_re = RandomEffects(df_re["lwage"], exog).fit(cov_type="robust")
re_coef = fit_re.params["union"]
re_se = fit_re.std_errors["union"]
print(f"Union coefficient: {re_coef:.4f}  (SE {re_se:.4f})")

**Interpretation.** RE returns 10.9 log points (SE 3.0), which sits between POLS (0.075) and FE (0.210). RE is mathematically a *weighted average* of the between and within estimators, with the weights determined by their relative variances. Because our data has very thin within variation in union status (only 9% of total), RE leans heavily toward the between picture and lands much closer to POLS than to FE. The RE standard error (0.030) is a striking 2.7× tighter than FE's (0.081), but that efficiency is real only if individual effects are uncorrelated with union membership.

## 12. The Hausman test: FE or RE?

The classic specification test for FE-vs-RE is due to **Hausman (1978)**. The intuition: if both estimators are consistent, they should give similar answers; if they differ a lot, the RE assumption is suspect and FE is preferred. Formally,

$$H = (\hat{\beta}_{\mathrm{FE}} - \hat{\beta}_{\mathrm{RE}})'\,[V_{\mathrm{FE}} - V_{\mathrm{RE}}]^{-1}\,(\hat{\beta}_{\mathrm{FE}} - \hat{\beta}_{\mathrm{RE}}) \sim \chi^2(k)$$

In words, this says: take the difference between the two coefficient vectors, weight it by the difference of the two variance matrices, and compare the resulting quadratic form to a chi-square distribution with degrees of freedom equal to the number of regressors. A large $H$ (small p-value) rejects the null that RE is consistent. Mapping to code: $\hat{\beta}_{\mathrm{FE}}$ is `fe_coef`, $\hat{\beta}_{\mathrm{RE}}$ is `re_coef`, and $V_{\mathrm{FE}}$ and $V_{\mathrm{RE}}$ are the squared standard errors.

In [ ]:
b_diff = np.array([fe_coef - re_coef])
v_diff = np.array([[fe_se ** 2 - re_se ** 2]])
H = float(b_diff @ np.linalg.pinv(v_diff) @ b_diff)
p_h = 1 - chi2.cdf(H, df=1)
print(f"H statistic: {H:.4f}   p-value = {p_h:.4f}")
print(f"β_FE − β_RE = {b_diff[0]:+.4f}")

**Interpretation.** The two estimators differ by about 0.101 log points; the test statistic is 1.79 on 1 degree of freedom, giving a p-value of 0.180. Conventionally, since 0.180 > 0.05, we *fail to reject* the null and conclude that RE is acceptable. But take this verdict with a grain of salt: the Hausman test has low power exactly when within variation is thin (our case: 9% within share for union). A noisy FE estimate inflates $V_{\mathrm{FE}}$ in the denominator and shrinks $H$, making non-rejection mechanical rather than substantive.

## 13. Correlated random effects (CRE / Mundlak): the modern bridge

**Mundlak (1978)** proposed a clever specification that bridges FE and RE. The idea: include each worker's *mean* of every time-varying regressor as an additional control, then run RE.

$$y_{it} = \alpha + \beta x_{it} + \gamma\,\bar{x}_i + u_{it}$$

In words, this says: model wages as a function of current union status, *plus* the worker's average union exposure across the panel. The coefficient $\beta$ on the time-varying $x_{it}$ captures the *within* effect — and Mundlak proved that under standard assumptions it is numerically identical to the FE coefficient. The coefficient $\gamma$ on the worker mean $\bar{x}_i$ captures the *between* effect of selection. If $\gamma \neq 0$, individual effects are correlated with union status and FE is preferred over RE. Mapping to code: $\beta$ is `cre_coef`, $\gamma$ is `mundlak_coef`, and $\bar{x}_i$ is the `union_bar` column we construct with `df.groupby("ID")["union"].transform("mean")`.

In [ ]:
# Stata: bysort ID: egen union_bar = mean(union); xtreg lwage union union_bar, re robust
df["union_bar"] = df.groupby("ID")["union"].transform("mean")
df_cre = df.set_index(["ID", "year"])
exog_cre = sm.add_constant(df_cre[["union", "union_bar"]])
fit_cre = RandomEffects(df_cre["lwage"], exog_cre).fit(cov_type="robust")
cre_coef = fit_cre.params["union"]
cre_se = fit_cre.std_errors["union"]
mundlak_coef = fit_cre.params["union_bar"]
mundlak_p = fit_cre.pvalues["union_bar"]
print(f"Union (within) coefficient: {cre_coef:.4f}  (SE {cre_se:.4f})")
print(f"Mundlak term (union_bar):   {mundlak_coef:+.4f}  (p = {mundlak_p:.4f})")
print(f"CRE within ≈ FE: {cre_coef:.4f} vs {fe_coef:.4f}")

**Interpretation.** The CRE union coefficient is 0.2103 — *exactly* the FE estimate to four decimal places, exactly as Mundlak's algebraic result predicts. The Mundlak term is −0.1441 with a p-value of 0.072, marginally non-significant at the 5% level but suggestive: workers with higher *average* union exposure tend to have lower wages even after conditioning on within-worker changes, which is consistent with negative selection into unionized jobs. The Mundlak signal points the same direction as the Hausman test but reaches the borderline-significant zone because it does not have to fight the same noise penalty.

## 14. Putting it all together: the method comparison

The figure below stacks all six basic estimators on a single chart with 95% confidence intervals.

In [ ]:
methods = ["POLS", "Between", "FDFE", "FE", "RE", "CRE"]
coefs = [pols_coef, between_coef, fdfe_coef, fe_coef, re_coef, cre_coef]
ses = [pols_se, between_se, fdfe_se, fe_se, re_se, cre_se]
colors = [METHOD_COLORS[m] for m in methods]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)
y_pos = np.arange(len(methods))
ci = [1.96 * s for s in ses]

ax.barh(y_pos, coefs, 0.6, xerr=ci, color=colors,
        edgecolor=DARK_NAVY, linewidth=0.5, capsize=4,
        error_kw={"ecolor": WHITE_TEXT, "capthick": 1.2})

for i, (c, s) in enumerate(zip(coefs, ses)):
    ax.text(c + 1.96 * s + 0.005, i, f"{c:.4f}", va="center",
            fontsize=10, color=LIGHT_TEXT)

ax.set_yticks(y_pos)
ax.set_yticklabels(methods, fontsize=13, color=LIGHT_TEXT)
ax.set_xlabel("Coefficient on Union", fontsize=13, color=LIGHT_TEXT)
ax.axvline(x=0, color=LIGHT_TEXT, linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_title("Effect of Union on Log Wages: Six Panel Estimators",
             fontsize=15, fontweight="bold", color=WHITE_TEXT)
ax.text(0.99, 0.02, f"Hausman: χ²={H:.2f}, p={p_h:.3f}",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=10, color=LIGHT_TEXT, style="italic")
plt.tight_layout()
plt.show()

# Comparison table
basic_comparison = pd.DataFrame({
    "method": methods,
    "coefficient": [round(c, 4) for c in coefs],
    "std_error": [round(s, 4) for s in ses],
})
print(basic_comparison.to_string(index=False))

**Interpretation.** The six methods cluster into two clear camps. The cross-sectional methods (POLS 0.075, Between 0.066, RE 0.109) report a union premium of 7–11 log points; the within methods (FDFE 0.211, FE 0.210, CRE 0.210) report 21 log points. The factor-of-three gap is the central pedagogical finding of this dataset and is consistent with a story in which unobserved worker ability correlates *negatively* with union status. Standard errors swing inversely: cross-sectional methods are 2–3× more precise but identify a different (and biased, under our hypothesis) parameter, while within methods are noisier but causally cleaner under weaker assumptions.

## 15. Adding controls: the extended models

Real applications usually include controls. We re-run POLS, TWFE, RE, and CRE with age, schooling, female, and year dummies on the right-hand side.

In [ ]:
df["age_bar"] = df.groupby("ID")["age"].transform("mean")

# POLS with controls
fit_pols_x = pf.feols(
    "lwage ~ union + age + schooling + female + C(year)",
    data=df, vcov="HC1")

# TWFE: schooling and female are time-invariant → absorbed by ID FE
fit_twfe_x = pf.feols("lwage ~ union + age | ID + year",
                      data=df, vcov={"CRV1": "ID"})

# RE + controls
df_rx = df.set_index(["ID", "year"])
exog_rx = sm.add_constant(df_rx[["union", "age", "schooling", "female"]])
fit_re_x = RandomEffects(df_rx["lwage"], exog_rx).fit(cov_type="robust")

# CRE + controls — adds within-means of time-varying regressors
exog_cx = sm.add_constant(
    df_rx[["union", "union_bar", "age", "age_bar", "schooling", "female"]])
fit_cre_x = RandomEffects(df_rx["lwage"], exog_cx).fit(cov_type="robust")

print(f"{'Variable':<11} {'POLS':>16} {'TWFE':>16} {'RE':>16} {'CRE':>16}")
print("=" * 76)
for var in ["union", "age", "schooling", "female"]:
    cells_str = []
    if var in fit_pols_x.coef().index:
        c, s = fit_pols_x.coef()[var], fit_pols_x.se()[var]
        cells_str.append(f"{c:>7.4f} ({s:.4f})")
    else:
        cells_str.append(f"{'—':>16}")
    if var in fit_twfe_x.coef().index:
        c, s = fit_twfe_x.coef()[var], fit_twfe_x.se()[var]
        cells_str.append(f"{c:>7.4f} ({s:.4f})")
    else:
        cells_str.append(f"{'absorbed':>16}")
    for fit in [fit_re_x, fit_cre_x]:
        if var in fit.params.index:
            c, s = fit.params[var], fit.std_errors[var]
            cells_str.append(f"{c:>7.4f} ({s:.4f})")
        else:
            cells_str.append(f"{'—':>16}")
    print(f"{var:<11} " + " ".join(cells_str))

**Interpretation.** Adding controls pulls the POLS union coefficient down to 0.057 — controls absorb some of the cross-sectional confounding — but TWFE and CRE still report a within-worker premium of about 0.21, leaving the four-camp gap (POLS 0.057 / RE 0.086 / TWFE 0.213 / CRE 0.210) largely intact. The schooling premium of 11.1% per year and the female penalty of 27.3 log points are stable across POLS, RE, and CRE because these regressors are essentially time-invariant; both are absorbed by individual FE in the TWFE column. The age coefficient is +0.021 in POLS, +0.022 in RE, and +0.033 in CRE, but flips to −0.058 in TWFE — a methodological artifact: with T = 2 and every worker aging by exactly two years between waves, age within an individual is collinear with the year dummy.

## 16. Discussion: what does our case study tell us?

We started with a deceptively simple question: does union membership raise wages, and if so by how much? Six estimators on the same dataset gave us answers ranging from 0.066 to 0.213 log points — a factor-of-three spread that is not noise but a structural feature of how the methods identify the parameter.

The cross-sectional camp (POLS, Between, RE) is asking *"how do union and non-union workers compare?"*. Their 7–11% answer is what we would report if we believed union members were comparable to non-members on every relevant unobservable. The within camp (FE, FDFE, TWFE, CRE) is asking *"what happens when the same worker switches union status?"*. Their 21% answer is what we would report if we trusted that nothing else changes for a worker between 2010 and 2012 except the things we observe. Both questions are legitimate; the gap between the answers is the empirical signature of selection on unobservables.

The Hausman test failed to reject the random-effects assumption (p = 0.180), which by the textbook script would tell us to use RE. But the test has low power exactly when within variation is thin, which is the case here (9% within share). The Mundlak alternative landed at p = 0.072 — borderline non-significant by a hair — and the Mundlak term itself was −0.144, suggesting that the workers with more union exposure are different (lower-paid on average) from workers with less. Both tests point in the same direction, but Mundlak's nuanced "almost significant" reading is more honest than Hausman's confident "fail to reject" verdict.

For a practitioner faced with this kind of dataset, the practical implication is that **CRE/Mundlak is usually the right specification to lead with**. It gives you the FE coefficient on the time-varying treatment (the within effect), the RE structure that lets you keep schooling and gender in the regression, and a built-in specification test (the t-statistic on the Mundlak term) that beats Hausman in low-power settings.

Stated formally in causal-inference language: the within estimators (FDFE, FE, TWFE, CRE) target the average treatment effect for *union switchers* — the subset of workers who actually changed union status between 2010 and 2012 — under the assumption of strict exogeneity conditional on the worker fixed effect. POLS and Between target a population-weighted association between union status and log wages and do not have a causal interpretation absent unconfoundedness.

## 17. Summary and next steps

**Takeaways.**

- **Method insight.** Three within recipes — first-differences, the within transformation, and dummy-variable FE — produce the same coefficient on union (0.2103, with FDFE differing by only +0.001 because of an intercept-driven year-trend artifact). This identity holds exactly when T = 2 and approximately when T > 2.
- **Data insight.** Almost all of our variation is between workers (union 94%, age 97%, schooling 100%). Only 9% of union variance is within. That is the slice of the data that fixed-effects estimators are working with, and it explains why FE standard errors (0.081) are 2.7× larger than RE standard errors (0.030).
- **Limitation.** With T = 2, our FE estimate is power-limited. The Hausman test fails to reject the RE assumption (p = 0.180) primarily because $V_{\mathrm{FE}}$ is large, not because RE is consistent. The Mundlak term tells the same story with more nuance (p = 0.072, borderline).
- **Next step.** A natural extension is to use all five waves of the panel (2010–2018) instead of just 2010 and 2012, which would give us T = 5 and dramatically more within variation in union status. With T > 2, the FD–FE gap becomes a real identification choice and event-study designs become possible.

## 18. Exercises

1. **Repeat the analysis with all five waves of the panel** (2010, 2012, 2014, 2016, 2018). How does the FE coefficient change? Does the Hausman test still fail to reject? What about the Mundlak term?
2. **Add an interaction with female.** Modify the FE specification to include `union × female` and interpret the coefficient. Does the union premium differ by gender?
3. **Try a clustered bootstrap.** Re-estimate the FE model with `vcov={"CRV1": "ID"}` and a wild cluster bootstrap (`pyfixest` supports `boot.iid()`). How do the bootstrap SEs compare to the analytical ones in this small-T setting?

## 19. References

1. [PyFixest documentation.](https://pyfixest.org/pyfixest.html)
2. [linearmodels: Panel models documentation.](https://bashtage.github.io/linearmodels/panel/introduction.html)
3. [scipy.stats.chi2 documentation.](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2.html)
4. [Wage panel dataset (`wage_panel_bob4.dta`) — quarcs-lab data-open repository.](https://github.com/quarcs-lab/data-open)
5. [Hausman, J. A. (1978). Specification Tests in Econometrics. *Econometrica*, 46(6), 1251–1271.](https://www.jstor.org/stable/1913827)
6. [Mundlak, Y. (1978). On the Pooling of Time Series and Cross Section Data. *Econometrica*, 46(1), 69–85.](https://www.jstor.org/stable/1913646)
7. [Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data*, 2nd ed. MIT Press.](https://mitpress.mit.edu/9780262232586/econometric-analysis-of-cross-section-and-panel-data/)